# 第 02 讲 Advanced：用术语表进行 LoRA 微调

这份 Notebook 在课程原版基础上修改了数据、提示词和评测，保留课程模型与固定依赖。

**第一次先运行第 0–2 节，看到 `train=124 / valid=12 / test=18` 后，检查问题与参考答案。**

随后进行：训练前评测 → LoRA 训练 → 同条件训练后评测 → 与查词对照比较 → 导出结果。

模型是 `Qwen/Qwen3.5-0.8B`，使用 T4 GPU、不量化的 LoRA。这里的 3 个 epoch 是预先确定的首次实验设置，数据量比原 FAQ 大；实际训练时间以本次输出为准。

原始 FAQ 练习的结果单独保留，不能与这里不同测试集上的分数直接作前后比较。当前 Notebook 未预填任何训练结果。

## 0. 新会话、T4 GPU 与课程依赖

选择 **运行时 → 更改运行时类型 → T4 GPU**。如果同一会话里已经加载过模型或进行过训练，先重新启动会话，再从本节开始。

In [ ]:
assert not any(name in globals() for name in ('model', 'tuned', 'trainer')), '请先重新启动会话，再从第 0 节开始。'
import subprocess
import torch
assert torch.cuda.is_available(), '请在运行时类型中选择 T4 GPU。'
print('GPU:', torch.cuda.get_device_name(0))
print('Colab torch:', torch.__version__)
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], check=True)

In [ ]:
# 沿用课程 PINS，使用 Colab 自带的 torch。
PINS = {'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0'}
spec = ' '.join(f'"{key}=={value}"' for key, value in PINS.items())
%pip uninstall -y -q torchao
%pip install -q {spec}
import transformers, peft, trl, accelerate, datasets
print({module.__name__: module.__version__ for module in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. 固定实验设置

使用全新的输出目录，避免混入原 FAQ 或其他次运行的文件。
训练前后使用相同提示词、18 道测试题、贪心生成和 32-bit 基础模型。

In [ ]:
import json, time, math, hashlib, csv, shutil, uuid
from datetime import datetime, timezone
from pathlib import Path
from transformers import set_seed
import pandas as pd
from IPython.display import display

CONFIG = {
    'base_model': 'Qwen/Qwen3.5-0.8B',
    'revision': '2fc06364715b967f1860aea9cf38778875588b17',
    'seed': 42, 'num_train_epochs': 3, 'warmup_steps': 6,
    'lr': 2e-4, 'lora_r': 16, 'lora_alpha': 32,
    'max_seq_len': 512, 'max_new_tokens': 192,
    'gen': {'do_sample': False},
}
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:6]
WORK = Path('/content/advanced_glossary') / RUN_ID
MATERIALS = WORK / 'materials'
MATERIALS.mkdir(parents=True, exist_ok=False)
set_seed(CONFIG['seed'])
pd.set_option('display.max_colwidth', None)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))
print('Output directory:', WORK)

## 2. 加载数据，检查拆分与检索对照

原始脚本产出 58 个术语 × 两种问法 = 116 条问答。这里为每条记录添加一致的系统提示，将定义包装为 JSON，再补充 8 条未收录术语的训练例，共 124 条。

验证集 12 条、测试集 18 条。问题文本没有跨集合重复；已知概念可以共享，考察的是新问法与覆盖范围判断。训练、验证和测试中的未收录术语分别选择。

**“未收录”指术语表没有这个条目，并不代表这个术语不存在，也不代表基础模型不知道它。** 本任务要求对这些问题请求复核。

模型评测时只接收问题和相同的系统提示，不提供术语定义或标准答案。检索对照可以访问术语表：按术语名/别名匹配，返回存储的定义；这种简单查词也可能漏掉改写问法。

In [ ]:
import urllib.request
import importlib.util
BASE_URL = 'https://raw.githubusercontent.com/sheaqiu/lecture-ai-engineering-2026-en/lec02/lectures/02/advanced/'
FILE_HASHES = {
    "README.md": "6defb544aa7e2e0c674c2400c6ca2238c1d8fbd345d7543b1a44536cc4b9987a",
    "advanced_glossary.py": "01ec16f695428e5bec9092d740a7f8c1d2121a1a16cd4b81589d5bed69096a5d",
    "glossary.json": "c3810badb201b5dd44dd9476189297a468567581b1368b82973791a0b709e9d5",
    "glossary_sft.jsonl": "57e84666ee5e263c51300b2ba8565481f437bc71288acebc76f72e413fa34d3f",
    "gold.json": "09a7ec647c834c11a559235c94fa9b30e9d22f5f61bc34ecf1dc5930f440dfd2",
    "manifest.json": "6722f5de4dca308aa95c86d88c8f02522cbdb63ab7790916f812c4409db063c7",
    "test.jsonl": "d754fc24b139f63bab6b46488792b83aa66d00cc96bfb0d7116d84bc51437ca4",
    "train.jsonl": "81c58283cbd84a6e322b22caa9565ccecba74849fa6a273f8926f3110d13b5e6",
    "valid.jsonl": "ba1c209907107db2abc50a141f2172f5ac2eb1e379ead93d0f8dcf82568a0f1f"
}
for name, expected_hash in FILE_HASHES.items():
    with urllib.request.urlopen(BASE_URL + name, timeout=60) as response:
        payload = response.read()
    assert hashlib.sha256(payload).hexdigest() == expected_hash, f'文件校验失败：{name}'
    (MATERIALS / name).write_bytes(payload)

module_spec = importlib.util.spec_from_file_location('aie_advanced', MATERIALS / 'advanced_glossary.py')
helper = importlib.util.module_from_spec(module_spec)
module_spec.loader.exec_module(helper)
glossary = json.loads((MATERIALS / 'glossary.json').read_text())
raw_records = [json.loads(line) for line in (MATERIALS / 'glossary_sft.jsonl').read_text().splitlines()]
prepared = helper.make_materials(glossary, raw_records)
train, valid, test, gold = [prepared[name] for name in ['train', 'valid', 'test', 'gold']]
for name in ['train', 'valid', 'test']:
    loaded = [json.loads(line) for line in (MATERIALS / f'{name}.jsonl').read_text().splitlines()]
    assert loaded == prepared[name]
assert json.loads((MATERIALS / 'gold.json').read_text()) == gold
print(f'glossary={len(glossary["terms"])} / raw={len(raw_records)}')
print(f'train={len(train)} / valid={len(valid)} / test={len(test)}')
print('训练样例：')
print(json.dumps(train[0], ensure_ascii=False, indent=2))
display(pd.DataFrame([{'Category': row['category'], 'Question': row['question'],
                      'Expected needs_review': gold[row['id']]['needs_review'],
                      'Reference answer': gold[row['id']]['reference_answer']} for row in test]))

In [ ]:
retrieval_summary, retrieval_rows = helper.evaluate(
    lambda question: helper.lookup_answer(question, glossary), test, gold, 'retrieval', WORK)
display(pd.DataFrame([retrieval_summary]).drop(columns=['by_category']))
display(pd.DataFrame([{'Question': row['question'], 'Lookup output': row['raw'], 'Content pass': row['content_ok']}
                      for row in retrieval_rows]))
print('第 0–2 节完成。先检查上面的训练样例、18 道题和参考答案，再继续第 3 节。')

## 3. 加载干净的基础模型并评测训练前表现

固定模型 revision，使用未量化的 32-bit 权重。先检查训练记录的 token 长度，确保没有因 512-token 上限截掉答案。

评分分开报告严格 JSON 格式和关键词内容。内容评分允许同一要点的若干常见词形，但不能替代人工检查；最后会导出全部原始回答。

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(CONFIG['base_model'], revision=CONFIG['revision'])
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
lengths = [len(tok.apply_chat_template(row['messages'], tokenize=True)) for row in train + valid]
assert max(lengths) <= CONFIG['max_seq_len'], f'训练记录过长：{max(lengths)}；应先检查数据与长度设置。'
TOKEN_LENGTHS = {'max_train_valid_tokens': max(lengths), 'limit': CONFIG['max_seq_len']}
print('Token lengths:', TOKEN_LENGTHS)
model = AutoModelForCausalLM.from_pretrained(
    CONFIG['base_model'], revision=CONFIG['revision'], dtype=torch.float32).to('cuda')
model.config.use_cache = True
model.eval()
print('Model weight memory (GB):', round(model.get_memory_footprint() / 1e9, 2))

def generate(active_model, question):
    prompt = tok.apply_chat_template(helper.messages(question), tokenize=False,
                                    add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors='pt', add_special_tokens=False).to(active_model.device)
    active_model.eval()
    with torch.no_grad():
        output = active_model.generate(**inputs, max_new_tokens=CONFIG['max_new_tokens'],
                                       pad_token_id=tok.pad_token_id, **CONFIG['gen'])
    return tok.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

before_summary, before_rows = helper.evaluate(lambda question: generate(model, question), test, gold, 'before', WORK)
display(pd.DataFrame([{'Question': row['question'], 'Before output': row['raw'],
                      'Format pass': row['format_ok'], 'Content pass': row['content_ok']} for row in before_rows]))
print(json.dumps(before_summary, ensure_ascii=False, indent=2))

## 4. LoRA 训练

使用 124 条训练记录、3 个 epoch。batch 4 × 梯度累积 2，预计每轮 16 次更新，共 48 次；以实际输出为准。

保留课程 LoRA 层选择和学习率。记录训练时间、全程平均训练 loss、最后一次日志窗口的 loss，以及训练结束后的验证 loss。验证集不参与梯度更新；本次没有根据测试结果选模型或调参。

如果训练中断，先保留日志并确认进度。完整重新实验应重新启动会话，从第 0 节开始，保证从干净的基础模型出发。

In [ ]:
from datasets import Dataset
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

set_seed(CONFIG['seed'])
train_ds, valid_ds = Dataset.from_list(train), Dataset.from_list(valid)
model.config.use_cache = False
lora = LoraConfig(r=CONFIG['lora_r'], lora_alpha=CONFIG['lora_alpha'], lora_dropout=0.05,
                  bias='none', task_type='CAUSAL_LM',
                  target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'])
sft = SFTConfig(output_dir=str(WORK / 'ckpt'), num_train_epochs=CONFIG['num_train_epochs'],
                per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG['lr'], lr_scheduler_type='cosine', warmup_steps=CONFIG['warmup_steps'],
                logging_steps=4, save_steps=16, save_total_limit=1,
                fp16=False, bf16=False, max_length=CONFIG['max_seq_len'], packing=False,
                report_to='none', seed=CONFIG['seed'])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=valid_ds,
                     peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()
torch.cuda.synchronize()
training_start = time.perf_counter()
train_output = trainer.train()
torch.cuda.synchronize()
training_seconds = time.perf_counter() - training_start
assert trainer.state.global_step > 0 and math.isfinite(float(train_output.training_loss)), '训练未正常完成。'
loss_logs = [row for row in trainer.state.log_history if 'loss' in row]
validation_metrics = trainer.evaluate()
TRAINING = {'seconds': round(training_seconds, 2), 'steps': trainer.state.global_step,
            'epochs': trainer.state.epoch, 'train_loss_average': float(train_output.training_loss),
            'last_logged_loss': loss_logs[-1]['loss'] if loss_logs else None,
            'last_loss_log_step': loss_logs[-1]['step'] if loss_logs else None,
            'validation_loss': validation_metrics.get('eval_loss'),
            'train_records': len(train), 'valid_records': len(valid)}
tuned = trainer.model
tuned.save_pretrained(WORK / 'adapter')
tok.save_pretrained(WORK / 'adapter')
tuned.config.use_cache = True
tuned.eval()
(WORK / 'training.json').write_text(json.dumps(TRAINING, indent=2), encoding='utf-8')
(WORK / 'training_log.json').write_text(json.dumps(trainer.state.log_history, indent=2), encoding='utf-8')
print(json.dumps(TRAINING, indent=2))

## 5. 同条件训练后评测与错误分析

不要改测试题、提示词或生成参数。模型变量 `model` 也已被注入 LoRA，因此这里直接使用之前保存的训练前输出；如需重新生成基础模型输出，使用 `with tuned.disable_adapter():`。

查看三个类别分别表现如何，以及哪些题训练后退步。检索对照的 JSON 格式由程序构造，因此其格式通过率不是模型能力；重点比较回答内容、覆盖处理、改写问题和维护方式。

In [ ]:
after_summary, after_rows = helper.evaluate(lambda question: generate(tuned, question), test, gold, 'after', WORK)
assert [row['id'] for row in before_rows] == [row['id'] for row in after_rows]
regressed = [before['question'] for before, after in zip(before_rows, after_rows)
             if before['both_ok'] and not after['both_ok']]
display(pd.DataFrame([before_summary, after_summary, retrieval_summary])[
    ['label', 'n', 'format_rate', 'content_rate', 'both_rate', 'seconds']])
display(pd.DataFrame([{'system': summary['label'], 'category': category, **scores}
                      for summary in [before_summary, after_summary, retrieval_summary]
                      for category, scores in summary['by_category'].items()]))
display(pd.DataFrame([{'Question': before['question'], 'Expected review': gold[before['id']]['needs_review'],
                      'Before output': before['raw'], 'After output': after['raw'],
                      'Before content pass': before['content_ok'], 'After content pass': after['content_ok']}
                     for before, after in zip(before_rows, after_rows)]))
print('Regressions:', regressed)

## 6. 导出结果并保留个人核验

导出 ZIP 包含数据、提示词、配置、三套逐题结果、训练日志、人工核验表和报告草稿。模型权重与检查点保留在运行时目录，未加入结果 ZIP。

下载后将 ZIP 交给 Codex 分析，并亲自检查原始回答。报告需要注明 AI 协助准备数据和代码，以及自己实际运行和核验的内容。

In [ ]:
source_manifest = json.loads((MATERIALS / 'manifest.json').read_text())
result = {
    'experiment': 'Lecture 02 Advanced: course glossary',
    'config': CONFIG,
    'versions': {module.__name__: module.__version__ for module in [transformers, peft, trl, accelerate, datasets, torch]},
    'method': 'LoRA (no quantization; float32 base weights)',
    'source_manifest': source_manifest, 'materials_sha256': FILE_HASHES,
    'prompt': helper.SYSTEM, 'token_lengths': TOKEN_LENGTHS,
    'before': before_summary, 'after': after_summary, 'retrieval': retrieval_summary,
    'regressed': regressed, 'training': TRAINING,
    'executed_training_myself': True,
    'manual_review_status': 'pending: inspect raw answers and complete manual_review.csv',
}
(WORK / 'result.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
review_rows = []
for before, after, lookup in zip(before_rows, after_rows, retrieval_rows):
    review_rows.append({'id': before['id'], 'category': before['category'], 'question': before['question'],
        'reference_answer': gold[before['id']]['reference_answer'], 'expected_needs_review': gold[before['id']]['needs_review'],
        'before_raw': before['raw'], 'after_raw': after['raw'], 'lookup_raw': lookup['raw'],
        'before_keyword_pass': before['content_ok'], 'after_keyword_pass': after['content_ok'],
        'lookup_keyword_pass': lookup['content_ok'], 'manual_verdict': '', 'reason': ''})
with (WORK / 'manual_review.csv').open('w', encoding='utf-8-sig', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(review_rows[0]))
    writer.writeheader()
    writer.writerows(review_rows)

report_lines = [
    '# Lecture 02 Advanced: draft for the Homework section', '',
    '- Level attempted: Advanced',
    '- Data: 58 course glossary entries; 116 original script records transformed into JSON answers, plus 8 negative training examples.',
    '- Splits: train=124, valid=12, test=18 (6 named terms, 6 paraphrases, 6 unlisted terms).',
    '- Neural answering is closed book. Unknown terms are defined by glossary membership, not by whether the base model knows the real-world concept.',
    '- Fixed 3 training epochs; rank=16; learning rate=2e-4; unquantized float32 base model.',
    '- AI assistance: Codex prepared the experiment code, question draft, and evaluation rules. Personal verification: TO COMPLETE.', '',
    '| System | Format rate | Keyword content rate | Both rate |',
    '|---|---:|---:|---:|',
]
for summary in [before_summary, after_summary, retrieval_summary]:
    report_lines.append(f"| {summary['label']} | {summary['format_rate']:.2%} | {summary['content_rate']:.2%} | {summary['both_rate']:.2%} |")
report_lines += ['', f"Training seconds: {TRAINING['seconds']}",
                 f"Average training loss: {TRAINING['train_loss_average']}",
                 f"Last logged loss (logging window, not last-example loss): {TRAINING['last_logged_loss']}",
                 f"Regressed questions: {json.dumps(regressed, ensure_ascii=False)}", '',
                 '## Complete after manually inspecting the original outputs', '',
                 '- Errors remaining after training, including review-flag mistakes: TO COMPLETE.',
                 '- Cases where the keyword scorer disagrees with semantic correctness: TO COMPLETE.',
                 '- Strengths and weaknesses compared with lexical glossary lookup: TO COMPLETE.',
                 '- When fine-tuning is suitable and when retrieval is more suitable: TO COMPLETE.',
                 '- Limits: small fixed test set, shared concepts across splits, keyword proxy scores, simple lexical baseline.',
                 '- Report the original FAQ classroom exercise separately in the Exercises section.', '']
(WORK / 'answer_draft.md').write_text('\n'.join(report_lines), encoding='utf-8')

import zipfile
archive = WORK.parent / f'advanced_results_{RUN_ID}.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(WORK.iterdir()):
        if path.is_file():
            bundle.write(path, path.name)
    for path in sorted(MATERIALS.iterdir()):
        if path.is_file():
            bundle.write(path, 'materials/' + path.name)
print('Results ZIP:', archive)
print('After downloading and confirming the ZIP is saved, disconnect and delete the Colab runtime.')
from google.colab import files
files.download(str(archive))